# LC 743 — Network Delay Time
**Day 64 | Pattern: Dijkstra's Algorithm**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Send a signal from node <em>k</em> and ask
"how long until ALL nodes receive it?" That is just
<em>max(shortest paths from k to every node)</em>.
Dijkstra's greedy min-heap gives us all shortest paths in one pass.
</div>

## Official Problem Statement

You are given a network of `n` nodes, labeled from `1` to `n`. You are also
given `times`, a list of travel times as directed edges
`times[i] = (u_i, v_i, w_i)`, where `u_i` is the source node, `v_i` is the
target node, and `w_i` is the time it takes for a signal to travel from
source to target.

We will send a signal from a given node `k`. Return the **minimum time** it
takes for all the `n` nodes to receive the signal. If it is impossible for
all the `n` nodes to receive the signal, return `-1`.

**Constraints:**
- `1 <= k <= n <= 100`
- `1 <= times.length <= 6000`
- `times[i].length == 3`
- `1 <= u_i, v_i <= n`, `u_i != v_i`
- `0 <= w_i <= 100`
- All `(u_i, v_i)` pairs are unique.

## What This Is Actually Asking

Model the network as a weighted directed graph and find the shortest path
from source `k` to every other node.
The answer is the largest of those shortest-path distances — because the
signal must travel along the slowest optimal route last.
If any node is unreachable (distance stays infinity), return `-1`.
This is a canonical single-source shortest path problem, solved optimally
with Dijkstra's algorithm and a min-heap.

## Walk Through an Example by Hand

```
times = [[2,1,1],[2,3,1],[3,4,1]], n=4, k=2

Graph (adjacency list):
  2 -> [(1, w=1), (3, w=1)]
  3 -> [(4, w=1)]

dist = [inf, inf, 0, inf, inf]   (index 0 unused, 1-based)
heap = [(0, 2)]                  (dist=0, start at node 2)

Pop (0, 2):
  relax edge 2->1: dist[1] = 0+1 = 1  push (1,1)
  relax edge 2->3: dist[3] = 0+1 = 1  push (1,3)
  heap = [(1,1),(1,3)]

Pop (1, 1):
  no outgoing edges from 1
  heap = [(1,3)]

Pop (1, 3):
  relax edge 3->4: dist[4] = 1+1 = 2  push (2,4)
  heap = [(2,4)]

Pop (2, 4):
  no outgoing edges from 4
  heap = []

dist = [inf, 1, 0, 1, 2]
max(dist[1..4]) = 2  => return 2
```

## The Picture

```
          w=1        w=1
   [2] --------> [1]   [4]
    |                    ^
    | w=1          w=1   |
    +--------> [3] ------+

  Dijkstra min-heap progression:

  Iteration | Pop      | dist updates
  ----------|----------|---------------------------
      1      | (0, k=2) | dist[1]=1, dist[3]=1
      2      | (1,  1)  | (no neighbours)
      3      | (1,  3)  | dist[4]=2
      4      | (2,  4)  | (no neighbours)

  Answer = max(dist[1..n]) = max(1, 0, 1, 2) = 2

  If any dist[i] == inf  =>  return -1
```

## When To Use This Pattern

- When you need the shortest path in a weighted graph with
  non-negative edges, think Dijkstra.
- When the problem asks for minimum cost/time to reach all nodes
  from a source, think Dijkstra + max of results.
- When the graph is sparse (edges << nodes^2), think adjacency list
  + min-heap Dijkstra over Floyd-Warshall.
- When a node is re-encountered in the heap at a higher distance,
  think skip-if-stale guard (`if d > dist[u]: continue`).
- When all edge weights are equal, think BFS instead of Dijkstra.

## The Approach

Build an adjacency list from `times`.  Initialise a `dist` array of size
`n+1` to infinity, then set `dist[k] = 0` and push `(0, k)` onto a
min-heap.
Pop the smallest `(d, u)` pair; skip if `d > dist[u]` (stale entry).
Otherwise relax every outgoing edge: if `d + w < dist[v]`, update `dist[v]`
and push the new entry.
After the heap empties, return `max(dist[1..n])` or `-1` if any value is
still infinity.

In [ ]:
import heapq
from collections import defaultdict
from typing import List

In [ ]:
# ── Test Harness ──────────────────────────────────────────────────────────

def test_harness(func):
    """
    Runs test cases against func(times, n, k).
    Prints PASSED / FAILED per case + final summary.
    """
    cases = [
        {
            "desc": "LC example 1 — 4 nodes, k=2",
            "times": [[2,1,1],[2,3,1],[3,4,1]],
            "n": 4, "k": 2,
            "expected": 2,
        },
        {
            "desc": "LC example 2 — 2 nodes, k=1",
            "times": [[1,2,1]],
            "n": 2, "k": 1,
            "expected": 1,
        },
        {
            "desc": "LC example 3 — 2 nodes, k=2 unreachable",
            "times": [[1,2,1]],
            "n": 2, "k": 2,
            "expected": -1,
        },
        {
            "desc": "single node",
            "times": [],
            "n": 1, "k": 1,
            "expected": 0,
        },
        {
            "desc": "longer chain with heavier edge",
            "times": [[1,2,4],[1,3,2],[3,2,1],[2,4,3]],
            "n": 4, "k": 1,
            "expected": 6,
        },
    ]

    passed = failed = 0
    for case in cases:
        result = func(
            case["times"], case["n"], case["k"]
        )
        ok = result == case["expected"]
        label = "PASSED" if ok else "FAILED"
        print(f"[{label}] {case['desc']}")
        if not ok:
            print(
                f"  got {result}, "
                f"expected {case['expected']}"
            )
            failed += 1
        else:
            passed += 1

    total = passed + failed
    print(f"\nResult: {passed}/{total} passed")

In [ ]:
# ── Solution Shell ────────────────────────────────────────────────────────

def networkDelayTime(
    times: List[List[int]], n: int, k: int
) -> int:
    """
    LC 743 — Network Delay Time

    Find the minimum time for a signal sent from node k to reach
    all n nodes. Return -1 if any node is unreachable.

    Strategy: Dijkstra's algorithm (single-source shortest path).

    Parameters
    ----------
    times : List[List[int]]
        Directed weighted edges [u, v, w].
    n : int
        Number of nodes (labeled 1..n).
    k : int
        Source node.

    Returns
    -------
    int
        Max shortest-path distance from k, or -1 if unreachable.

    Time  : O((V + E) log V)
    Space : O(V + E)

    Trace
    -----
    times=[[2,1,1],[2,3,1],[3,4,1]], n=4, k=2
    -> dist = [inf,1,0,1,2]  -> max = 2
    """
    # TODO: build adjacency list
    print(f"[DEBUG] n={n}, k={k}, edges={len(times)}")

    # TODO: initialise dist array (size n+1, index 0 unused)

    # TODO: min-heap Dijkstra
    #   heap = [(0, k)]
    #   while heap:
    #       d, u = heapq.heappop(heap)
    #       if d > dist[u]: continue
    #       for v, w in adj[u]:
    #           if d + w < dist[v]: ...

    # TODO: return max(dist[1..n]) or -1 if any == inf
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(networkDelayTime)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (Bellman-Ford) | O(V * E) | O(V) |
| Floyd-Warshall (all pairs) | O(V^3) | O(V^2) |
| **Dijkstra + min-heap (optimal)** | **O((V+E) log V)** | **O(V+E)** |

Where **V** = nodes (≤100), **E** = edges (≤6000).

The skip-if-stale guard (`if d > dist[u]: continue`) keeps the heap
clean without a decrease-key operation, making this easy to implement
with Python's `heapq`.

## Real World Connection

At **Citi**, inter-datacenter message routing uses shortest-path logic to
minimise latency for trade confirmations crossing multiple network hops.
In **AWS**, services like Global Accelerator and Route 53 apply Dijkstra-
style algorithms to find the lowest-latency edge location for a user
request across a worldwide network of nodes.
For **Data Engineering**, Spark's DAG scheduler implicitly solves a
shortest-path problem when determining the optimal execution plan across
a cluster, minimising shuffle and data-transfer cost.
Pipeline dependency graphs in tools like Apache Airflow also rely on
shortest-path traversal to determine critical paths and estimate
completion time for long-running workflows.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra